
This notebook does:
- load airborne hyperspectral data from `data/you-shall-not-pass/Obrazy lotniczne`
- display false-color composite from the cube
- compute water quality metrics (Chl-a, DOC, turbidity) from airborne data
- download Sentinel-2 patch close to airborne acquisition date
- compute same indices on Sentinel-2 and compare

In [ ]:
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import rasterio
from rasterio.plot import reshape_as_raster, reshape_as_image
from datetime import datetime, timedelta

BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), '..', 'data', 'you-shall-not-pass', 'Obrazy lotniczne'))
print('base path', BASE_DIR)

In [ ]:
hdrs = glob(os.path.join(base_dir, '*.hdr'))
bsqs = glob(os.path.join(base_dir, '*.bsq'))

hdr_files, bsq_files = list_hyperspectral_files()
print('hdr:', hdr_files)
print('bsq:', bsq_files)

In [ ]:
meta = {}
with open(hdr_path, 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        if '=' in line:
            key, val = line.split('=', 1)
            key, val = key.strip().lower(), val.strip()
            meta[key] = val

# select first pair
if hdr_files and bsq_files:
    hdr_path = hdr_files[0]
    raw_path = None
    for p in bsq_files:
        if os.path.splitext(os.path.basename(p))[0] == os.path.splitext(os.path.basename(hdr_path))[0]:
            raw_path = p
            break
    print('selected', hdr_path, raw_path)
    meta = read_envi_hdr(hdr_path)
    print(meta.get('samples'), meta.get('lines'), meta.get('bands'))
else:
    raise FileNotFoundError('No hyperspectral files found')

In [ ]:
samples = int(meta['samples'])
lines = int(meta['lines'])
bands = int(meta['bands'])
interleave = meta.get('interleave', 'bsq').lower()
dtype = np.dtype('float32') if 'float' in meta.get('data type', '4') else np.dtype('uint16')
wavelengths = None
if 'wavelength' in meta:
    w = meta['wavelength'].strip('{} ').split(',')
    wavelengths = np.array([float(x) for x in w if x.strip()])

samples, lines, bands, interleave, dtype, wavelengths = parse_envi_header(meta)
print(samples, lines, bands, interleave, dtype, wavelengths[:5] if wavelengths is not None else None)

In [ ]:
arr = np.fromfile(raw_path, dtype=dtype)
arr = arr.reshape((bands, lines, samples))

cube = read_bsq(raw_path, samples, lines, bands, dtype)
print('cube shape', cube.shape)

In [ ]:
rgb = np.stack([img[r_band], img[g_band], img[b_band]], axis=-1)
rgb = np.clip(rgb, clip[0], clip[1])
rgb = (rgb - clip[0]) / (clip[1] - clip[0])

# choose hypothetical visible bands by wavelength or index
if wavelengths is not None:
    r_idx = np.argmin(np.abs(wavelengths - 670))
    g_idx = np.argmin(np.abs(wavelengths - 560))
    b_idx = np.argmin(np.abs(wavelengths - 490))
else:
    r_idx, g_idx, b_idx = 80, 40, 20

false_color = to_rgb_index(cube, r_idx, g_idx, b_idx, clip=(0, 0.3))
plt.figure(figsize=(8, 8)); plt.imshow(false_color); plt.title('airborne RGB false color'); plt.axis('off')

In [ ]:
# Visible band indices may need tuning. For these formulas, provide the chosen band indices.

def safe_ratio(n, d, eps=1e-6):
    return n / (d + eps)

# choose bands from wavelengths or indices
chl_band = np.argmin(np.abs(wavelengths - 705)) if wavelengths is not None else 85
ref_band = np.argmin(np.abs(wavelengths - 665)) if wavelengths is not None else 80

chla = safe_ratio(cube[chl_band], cube[ref_band])

doc = safe_ratio(cube[np.argmin(np.abs(wavelengths - 560)) if wavelengths is not None else 45], cube[np.argmin(np.abs(wavelengths - 665)) if wavelengths is not None else 80])

turbidity = cube[np.argmin(np.abs(wavelengths - 700)) if wavelengths is not None else 75]

print('chla mean', np.nanmean(chla), 'doc mean', np.nanmean(doc), 'turbidity mean', np.nanmean(turbidity))

for label, data in [('Chl-a', chla), ('DOC', doc), ('Turbidity', turbidity)]:
    plt.figure(figsize=(6, 4)); plt.imshow(data, cmap='viridis'); plt.colorbar(); plt.title(label)

In [ ]:
x = samples // 2
y = lines // 2
print('sample index', x, 'line index', y)

spec = cube[:, y, x]
plt.figure(figsize=(8, 3)); plt.plot(wavelengths if wavelengths is not None else np.arange(bands), spec)
plt.title('Spectral signature at central pixel'); plt.xlabel('wavelength'); plt.ylabel('reflectance')

In [ ]:
from sentinelsat import SentinelAPI, read_geojson, geojson_to_wkt

# replace with valid username and password
api = SentinelAPI('MY_USERNAME', 'MY_PASSWORD', 'https://scihub.copernicus.eu/dhus')

# get acquisition date from airborne metadata if possible
acq_date = datetime.strptime('2024-06-01', '%Y-%m-%d')  # adjust by actual date
footprint = 'POLYGON((...))'  # replace with ROI coords

products = api.query(footprint,
                     date=(acq_date - timedelta(days=7), acq_date + timedelta(days=7)),
                     platformname='Sentinel-2',
                     processinglevel='Level-2A',
                     cloudcoverpercentage=(0, 30))

print('found', len(products), 'products')

# Pick best candidate and download
if products:
    prod_id, prod_info = next(iter(sorted(products.items(), key=lambda t: t[1]['cloudcoverpercentage'])))
    print('downloading', prod_info['title'])
    api.download(prod_id, directory_path='sentinel2_download')
else:
    print('no products found; please adjust footprint/date')

In [ ]:
import glob

s2_dir = 'sentinel2_download'
# locate band files B04 (red), B03 (green), B02 (blue), B08 (nir), B11 (swir)
pattern = os.path.join(s2_dir, '**', '*_B0[2348][A]?.tif')
files = glob.glob(pattern, recursive=True)
print('band files', files)

# for this example we assume we have the 10m bands B02, B03, B04, B08
band_map = {
    'B02': None,
    'B03': None,
    'B04': None,
    'B08': None,
    'B11': None,
}
for f in glob.glob(os.path.join(s2_dir, '**', '*.jp2'), recursive=True):
    base = os.path.basename(f)
    for b in band_map.keys():
        if f'_{b}_' in base or f'_{b}.jp2' in base:
            band_map[b] = f

print('band_map', band_map)

# load bands into arrays and to 10m res if needed
# simplification: use B04,B03,B02,B08 at same resolution (if they are 10m) 
# (if 20m, resample or use other formula)

band_data = {}
for b, p in band_map.items():
    if p is None:
        continue
    with rasterio.open(p) as src:
        band_data[b] = src.read(1).astype('float32') / 10000.0

if not band_data:
    raise FileNotFoundError('Sentinel-2 bands not found in downloaded archive')

chl_s2 = safe_ratio(band_data['B08'], band_data['B04'])
doc_s2 = safe_ratio(band_data['B03'], band_data['B04'])
turb_s2 = safe_ratio(band_data['B11'], band_data['B08'])

for label, data in [('S2 Chl-a', chl_s2), ('S2 DOC', doc_s2), ('S2 Turbidity', turb_s2)]:
    plt.figure(figsize=(6, 4)); plt.imshow(data, cmap='viridis'); plt.colorbar(); plt.title(label)

In [ ]:
# Here we compare in pixel coordinates approx mid-point.

air_x, air_y = x, y
s2_x, s2_y = chl_s2.shape[1] // 2, chl_s2.shape[0] // 2

print('airborne indices at center:',
      'chla', chla[air_y, air_x],
      'doc', doc[air_y, air_x],
      'turb', turbidity[air_y, air_x])
print('sentinel2 indices at center:',
      'chla', chl_s2[s2_y, s2_x],
      'doc', doc_s2[s2_y, s2_x],
      'turb', turb_s2[s2_y, s2_x])

# scatter comparison at coarse sample
air_idx_vals = np.column_stack((chla.flatten(), doc.flatten(), turbidity.flatten()))
s2_idx_vals = np.column_stack((chl_s2.flatten()[:air_idx_vals.shape[0]],
                               doc_s2.flatten()[:air_idx_vals.shape[0]],
                               turb_s2.flatten()[:air_idx_vals.shape[0]]))

plt.figure(figsize=(10, 3))
for i, name in enumerate(['Chl-a', 'DOC', 'Turbidity']):
    plt.subplot(1, 3, i + 1)
    plt.scatter(air_idx_vals[:, i], s2_idx_vals[:, i], s=1, alpha=0.2)
    plt.xlabel('airborne')
    plt.ylabel('Sentinel-2')
    plt.title(name)
plt.tight_layout()

- set correct ROI footprint and acquisition date.
- for accurate water quality indices use published formulas and validation data.
- if `sentinelsat` is not installed, run `pip install sentinelsat`.